# 17: Estado estruturado (cada agente no seu campo)

Até aqui, o Supervisor decidia o próximo passo **contando mensagens pelo nome**. Havia uma mensagem do YFinance e
nenhuma do Analyzer? Etapa 2, notícias da empresa. Uma mensagem do Analyzer? Etapa 3, o setor. Duas? Etapa 4. O
Consolidador fazia a mesma conta para escrever as limitações ("as notícias não foram buscadas").

Isso funciona enquanto cada agente fala exatamente uma vez e na ordem certa. Os próximos passos do projeto quebram
as duas condições:

- **Agentes em paralelo** (o `Send` do LangGraph): o Analyzer e uma pesquisa rodando ao mesmo tempo terminam em
  qualquer ordem, e a posição das mensagens deixa de dizer quem fez o quê.
- **Retry**: um agente que falha e roda de novo deixa **duas** mensagens. O guard conta duas passagens do Analyzer
  e pula a etapa do setor, que nunca foi feita.

A solução é dar a cada saída um lugar próprio no state. O nó do YFinance grava `dados_mercado`, a primeira
passagem do Analyzer grava `noticias`, as seguintes gravam `contexto_setor` e o Investigator grava `diagnostico`.
Os guards e o Consolidador passam a perguntar "este campo está preenchido?", e não "quantas mensagens com este
nome existem?". Para quem usa, nada muda: o caminho, as mensagens e o relatório são os mesmos.

In [1]:
import sys
import warnings
from pathlib import Path

# Sobe pastas até achar a raiz do projeto (onde está o pyproject.toml), de onde quer que o notebook seja aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ))

from dotenv import load_dotenv
_ = load_dotenv(RAIZ / ".env")

# Aviso de depreciação interno do LangGraph, sem relação com o projeto: só polui a saída.
# O filtro vem depois do import do langchain_core, que instala o filtro dele ao ser importado.
from langchain_core._api.deprecation import LangChainPendingDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainPendingDeprecationWarning)
warnings.filterwarnings("ignore", message="Function wrap_gemini is in beta")

## 1. Conceitos

**State do LangGraph:** o dicionário que passa de nó em nó. Cada nó recebe o state inteiro e devolve só as chaves
que quer mudar. O LangGraph junta essa resposta ao state antes de chamar o próximo nó.

**TypedDict:** a classe que descreve o state (`MessagesState`, em `shared/types.py`). Ela diz quais chaves existem
e o tipo de cada uma, mas não obriga nenhuma a estar presente: no primeiro passo o state tem só `messages`. Por
isso todo acesso aos campos novos é `state.get(...)`, que devolve `None` quando o campo não existe.

**Reducer:** a regra que diz como juntar o valor novo com o antigo. Cada chave escolhe a sua:

| Chave | Reducer | Efeito |
|---|---|---|
| `messages` | `add_messages` | concatena: a mensagem nova entra no fim do histórico |
| `loop_count` | `operator.add` | soma: cada passagem do Supervisor devolve 1 |
| as demais (`dados_mercado`, `noticias`...) | nenhum (padrão) | sobrescreve: vale o último valor escrito |

**Por que `messages` continua:** os agentes ainda recebem o histórico inteiro como contexto (o Analyzer lê os
dados do YFinance nas mensagens), o Consolidador monta os insumos do LLM a partir delas e o dashboard mostra a
conversa. Os campos novos são uma cópia do texto final de cada agente, guardada num lugar fixo para **decidir**.

**Sinais de anomalia (o que aciona o Investigator):** regras em código sobre os números da empresa. Queda do lucro
líquido de mais de 30% contra o **mesmo trimestre do ano anterior** (comparar com o ano anterior tira o efeito da
sazonalidade), prejuízo no último trimestre e Piotroski F-Score de 3 ou menos (uma nota de 0 a 9 sobre a saúde
financeira: lucro, caixa, endividamento, eficiência). O notebook 10 mostra as regras rodando.

## 2. Os campos e seus reducers, lidos da própria classe

`get_type_hints` com `include_extras=True` mostra o `Annotated` de cada chave: quando há um reducer, ele aparece
ali. As chaves sem reducer usam o padrão, que sobrescreve.

In [2]:
import typing

from fluxo_financeiro.shared.types import MessagesState

dicas = typing.get_type_hints(MessagesState, include_extras=True)
for chave, tipo in dicas.items():
    extras = getattr(tipo, "__metadata__", ())
    reducer = extras[0].__name__ if extras else "(padrão: sobrescreve)"
    print(f"{chave:22} {reducer}")

messages               _add_messages
next_agent             (padrão: sobrescreve)
loop_count             add
supervisor_reasoning   (padrão: sobrescreve)
decidido_por           (padrão: sobrescreve)
sinais_anomalia        (padrão: sobrescreve)
relatorio              (padrão: sobrescreve)
dados_mercado          (padrão: sobrescreve)
noticias               (padrão: sobrescreve)
contexto_setor         (padrão: sobrescreve)
diagnostico            (padrão: sobrescreve)


## 3. Sobrescrever x somar, num grafo de brinquedo

Um grafo mínimo, sem LLM, com dois nós que escrevem nas mesmas chaves. `contador` tem `operator.add`; `texto` não
tem reducer.

In [3]:
import operator
from typing import Annotated, Optional, TypedDict

from langgraph.graph import END, START, StateGraph


class Brinquedo(TypedDict):
    contador: Annotated[int, operator.add]
    texto: Optional[str]


def primeiro(state):
    return {"contador": 1, "texto": "escrito pelo primeiro nó"}


def segundo(state):
    print("o segundo nó leu:", state)
    return {"contador": 1, "texto": "escrito pelo segundo nó"}


g = StateGraph(Brinquedo)
g.add_node("primeiro", primeiro)
g.add_node("segundo", segundo)
g.add_edge(START, "primeiro")
g.add_edge("primeiro", "segundo")
g.add_edge("segundo", END)

print("estado final:  ", g.compile().invoke({"contador": 0}))

o segundo nó leu: {'contador': 1, 'texto': 'escrito pelo primeiro nó'}
estado final:   {'contador': 2, 'texto': 'escrito pelo segundo nó'}


## 4. O caso que enganava o guard

Um estado montado à mão, como se o Analyzer tivesse rodado **duas vezes para a mesma tarefa** (um retry): duas
mensagens com `name="Analyzer"`, mas só a das notícias da empresa foi gravada no campo (`noticias`). O setor nunca
foi buscado, então `contexto_setor` está vazio.

A função `contar_como_antes` reconstrói, só para comparar, a regra antiga (ela não existe mais no pacote): contar
as mensagens do Analyzer. Depois chamamos o `supervisor_node` de verdade.

In [4]:
from langchain_core.messages import HumanMessage

from fluxo_financeiro.graph.supervisor import supervisor_node

pedido = HumanMessage("Analise a ação VALE3")
dados = HumanMessage("Dados para VALE3.SA: ...", name="YFinance")
noticias_1 = HumanMessage("Notícias da empresa...", name="Analyzer")
noticias_2 = HumanMessage("Notícias da empresa (de novo, depois de um retry)...", name="Analyzer")

estado = {
    "messages": [pedido, dados, noticias_1, noticias_2],
    "loop_count": 3,
    "sinais_anomalia": [],
    "dados_mercado": dados.text,
    "noticias": noticias_2.text,
}


def contar_como_antes(messages):
    """Reconstrução da regra antiga, só para comparação: decidia pelo número de mensagens do Analyzer."""
    n = sum(1 for m in messages if m.name == "Analyzer")
    if n == 0:
        return "Analyzer (etapa 2, notícias)"
    if n == 1:
        return "Analyzer (etapa 3, setor)"
    return "etapa 4 (setor dado como feito)"


print("Mensagens do Analyzer:", sum(1 for m in estado["messages"] if m.name == "Analyzer"))
print("Regra antiga (contando):", contar_como_antes(estado["messages"]))

decisao = supervisor_node(estado)
print(f"Regra nova (campos):     {decisao['next_agent']} (decidido por: {decisao['decidido_por']})")
print("Tarefa:", decisao["messages"][0].text[:160], "...")

2026-11-08 17:47:13,535 - fluxo_financeiro - INFO - [supervisor] inicio — loop_count=3


2026-11-08 17:47:13,535 - fluxo_financeiro - INFO - [supervisor] guard: noticias sem contexto_setor → forcando Analyzer (etapa 3: setor)


Mensagens do Analyzer: 2
Regra antiga (contando): etapa 4 (setor dado como feito)
Regra nova (campos):     Analyzer (decidido por: guard)
Tarefa: Busque informações sobre o setor de atuação da empresa (VALE3), usando o setor e subsetor (ou a indústria) informados nos dados do YFinance, e o posicionamento  ...


`decidido por: guard` é a prova de que o LLM não foi consultado: o guard leu `noticias` preenchido e
`contexto_setor` vazio e mandou para o setor. A contagem teria pulado direto para a Etapa 4.

## 5. Uma análise real da WEGE3

O pipeline inteiro, uma vez (cerca de 30 segundos, algumas chamadas ao Gemini). O `stream` devolve o que cada nó
mudou; a cada decisão do Supervisor, a célula mostra quais campos já estavam preenchidos naquele momento e qual
regra decidiu.

In [5]:
from fluxo_financeiro import app

CAMPOS = ("dados_mercado", "noticias", "contexto_setor", "diagnostico")


def regra(antes, update):
    """Qual condição sobre os campos explica a decisão (a mesma ordem dos guards no supervisor_node)."""
    if update["decidido_por"] == "hard_stop":
        return "loop_count chegou a 6"
    if update["decidido_por"] == "llm":
        return "nenhum guard se aplicou: o LLM decidiu"
    if update["next_agent"] == "FINISH":
        return "diagnostico preenchido"
    if update["next_agent"] == "Investigator":
        return "noticias e contexto_setor preenchidos, sinais_anomalia não vazio"
    if "noticias" in antes:
        return "noticias preenchido, contexto_setor vazio (etapa 3)"
    return "dados_mercado preenchido, noticias vazio (etapa 2)"


preenchidos = set()
final = None
for modo, pedaco in app.stream({"messages": [HumanMessage("Analise a ação WEGE3")]}, stream_mode=["updates", "values"]):
    if modo == "values":
        final = pedaco
        continue
    for no, update in pedaco.items():
        if no == "Supervisor":
            antes = sorted(preenchidos)
            print(f"Supervisor -> {update['next_agent']:12} por {update['decidido_por']:9} | campos: {antes or 'nenhum'}")
            print(f"   regra: {regra(antes, update)}")
        else:
            novos = [c for c in CAMPOS if update.get(c) is not None]
            preenchidos.update(novos)
            print(f"{no:12} gravou: {novos or 'nenhum campo de agente'}")

2026-11-08 17:47:13,545 - fluxo_financeiro - INFO - [supervisor] inicio — loop_count=0


2026-11-08 17:47:14,793 - fluxo_financeiro - INFO - [supervisor] decisao=YFinance | tarefa='Colete os dados financeiros, métricas e histórico de preços para a ação WEGE3.'


2026-11-08 17:47:14,794 - fluxo_financeiro - DEBUG - [router] YFinance → YFinance


2026-11-08 17:47:14,796 - fluxo_financeiro - INFO - [node] YFinance: invocando agente (2 msgs)


Supervisor -> YFinance     por llm       | campos: nenhum
   regra: nenhum guard se aplicou: o LLM decidiu


2026-11-08 17:47:15,844 - fluxo_financeiro - INFO - [tool] buscar_dados_acao_b3: ticker=WEGE3.SA


... (34 outputs left out)

2026-11-08 17:47:36,547 - fluxo_financeiro - INFO - [node] Consolidador: consolidando (7 msgs)


Supervisor -> FINISH       por llm       | campos: ['contexto_setor', 'dados_mercado', 'noticias']
   regra: nenhum guard se aplicou: o LLM decidiu


2026-11-08 17:47:38,929 - fluxo_financeiro - INFO - [node] Consolidador: relatório pronto (1730 chars)


Consolidador gravou: nenhum campo de agente


## 6. Os quatro campos no state final

In [6]:
for campo in CAMPOS:
    valor = final.get(campo)
    if valor is None:
        print(f"{campo:15} None")
    else:
        print(f"{campo:15} {len(valor):5} caracteres | {valor[:90]!r}...")

print()
print("loop_count:", final["loop_count"])
print("Limitações do relatório:", final["relatorio"].limitacoes if final.get("relatorio") else "relatório de emergência")

dados_mercado    3353 caracteres | 'Dados para WEGE3.SA:\nEmpresa: WEG SA\nPreço Atual: R$ 48.19 (no dia: -0.23%)\nFaixa de 52 se'...
noticias         1244 caracteres | 'A ação da **WEG (WEGE3)** é negociada atualmente a R$ 48,19, cotada próxima à sua mínima r'...
contexto_setor   1373 caracteres | 'A WEG (**WEGE3**) lidera com ampla vantagem o subsetor de Máquinas e Equipamentos na B3, o'...
diagnostico     None

loop_count: 4
Limitações do relatório: []


## 7. O campo é uma cópia, não um resumo

O texto em cada campo é o mesmo da mensagem do agente. A mensagem continua no histórico para os agentes lerem; o
campo existe para o código decidir.

In [7]:
ultima = {}
for m in final["messages"]:
    if m.name in ("YFinance", "Analyzer", "Investigator"):
        ultima.setdefault(m.name, []).append(m.text)

print("dados_mercado == mensagem do YFinance:", final["dados_mercado"] == ultima["YFinance"][0])
print("noticias == 1a mensagem do Analyzer:", final["noticias"] == ultima["Analyzer"][0])
print("contexto_setor == última mensagem do Analyzer:", final["contexto_setor"] == ultima["Analyzer"][-1])
if "Investigator" in ultima:
    print("diagnostico == mensagem do Investigator:", final["diagnostico"] == ultima["Investigator"][0])

dados_mercado == mensagem do YFinance: True
noticias == 1a mensagem do Analyzer: True
contexto_setor == última mensagem do Analyzer: True


## O que observar

- Só duas chaves do `MessagesState` têm reducer: `messages` (`_add_messages`, concatena) e `loop_count` (`add`, soma). Todas as outras, inclusive os quatro campos novos, sobrescrevem.
- No grafo de brinquedo, o segundo nó leu o que o primeiro escreveu. No fim, `contador` virou 2 (somou) e `texto` ficou só com o valor do segundo nó (sobrescreveu).
- No estado com duas mensagens do Analyzer, a contagem antiga mandaria para a Etapa 4 com o setor nunca buscado. O guard novo leu `noticias` preenchido e `contexto_setor` vazio e mandou para o setor, decidido por guard, sem chamar o LLM.
- Na WEGE3, o primeiro passo não tinha campo nenhum (o state começa só com `messages`): nenhum guard se aplicou e o LLM mandou para o YFinance. Ler um campo ausente com `state.get` devolve `None`, sem erro.
- As Etapas 2 e 3 foram decididas por guard, cada uma lendo um campo: `dados_mercado` sem `noticias`, depois `noticias` sem `contexto_setor`.
- A WEGE3 não teve sinal de anomalia, então o guard da Etapa 4 não se aplicou e o LLM escolheu FINISH. Por isso `diagnostico` ficou `None` e o `loop_count` terminou em 4. A VALE3 costuma seguir o outro caminho (notebook 10).
- Cada nó gravou só o seu campo, e o texto do campo é idêntico ao da mensagem. O Consolidador não grava campo de agente: ele lê os campos para escrever as limitações (nesta execução, nenhuma).
- Nesta execução o Banco Central não respondeu a série do CDI (o traceback no log). A falha ficou na tool, o bloco seguiu com as séries que responderam e a análise não parou.
- Os números da análise (preço, textos das notícias) mudam a cada execução. O que não muda é qual guard lê qual campo.